# IIP314W Optimización Aplicada a Negocios - 2026-T1
## Ayudantía 3: Condiciones Karush-Kuhn-Tucker (KKT)

### 1. Resumen Teórico Rápido
Recordemos las **Condiciones KKT** para un problema en forma estándar (minimización con restricciones $\le 0$):
*   **Estacionariedad:** El gradiente de la función objetivo en el óptimo, más la combinación lineal de los gradientes de las restricciones, debe ser cero.
*   **Factibilidad Primal:** Se deben cumplir todas las restricciones originales del problema.
*   **Holgura Complementaria:** $\mu_i g_i(x^*) = 0$. Los multiplicadores de las restricciones inactivas (donde $g_i < 0$) deben ser obligatoriamente $0$.
*   **Multiplicadores No Negativos:** $\mu_i \ge 0$. Para restricciones tipo $\le 0$ en minimización, esto garantiza que la dirección del gradiente apunte hacia "adentro" o a lo largo del cono tangente a la región factible localmente.

### 2. Ejercicio Práctico: Producción Óptima

**Contexto:**
Una empresa de tecnología fabrica dos tipos de servidores ($x_1$ de gama media y $x_2$ de alta gama). Debido a los contratos por componentes limitados y la reducción de precios para altos volúmenes, la función de utilidades de la empresa (medida en miles de dólares) al fabricar estos recursos es **no lineal**:
$$ \max U(x_1, x_2) = 80x_1 - 2x_1^2 + 100x_2 - x_2^2 $$

La empresa no puede producir infinitamente, está sujeta a las siguientes 4 restricciones operativas:
1.  **Capacidad de ensamblaje general (horas base):** Las horas combinadas limitan la producción a: $x_1 + x_2 \le 40$.
2.  **Capacidad de pruebas exhaustivas térmicas:** El proceso térmico en componentes alta gama restringe así la línea: $x_1 + 3x_2 \le 90$.
3.  **Suministro de placas madre:** El suministro de placas base tipo M impone una cuota de inventario: $2x_1 + x_2 \le 65$.
4.  **Demanda máxima tope:** La demanda de la empresa compradora para servidores gama media tiene un límite de cuota absoluto (contrato): $x_1 \le 30$.
5.  **Condiciones lógicas de no negatividad:** $x_1, x_2 \ge 0$.

Se pide:
**a)** Plantear el problema en **forma estándar**.
**b)** Plantear todas las ecuaciones necesarias para resolverlo con **KKT analíticamente** a mano y explicar cómo probaría algebraicamente el punto $x_1=15, x_2=25$ como candidato a óptimo (considera usar cuáles restricciones son activas).
**c)** Utilizar `scipy.optimize.minimize` para resolverlo computacionalmente, y verificar al final el valor real de sus holguras o "slacks" para ver cuáles terminaron en el borde (activas).

#### **a) Forma Estándar**
Para utilizar la forma estándar de análisis (donde aplicamos KKT), transformamos el problema de maximización convirtiéndolo a minimización, recordando que $\max U(x) \equiv \min -U(x)$. E indicamos las restricciones hacia una inecuación de desigualdad del tipo $g(x) \le 0$.

Función Objetivo:
$$ \min f(x_1, x_2) = 2x_1^2 - 80x_1 + x_2^2 - 100x_2 $$

Sujeto a:
*   $g_1(x): x_1 + x_2 - 40 \le 0$
*   $g_2(x): x_1 + 3x_2 - 90 \le 0$
*   $g_3(x): 2x_1 + x_2 - 65 \le 0$
*   $g_4(x): x_1 - 30 \le 0$

(Anexando las clásicas No Negativas)
*   $g_5(x): -x_1 \le 0$
*   $g_6(x): -x_2 \le 0$

#### **b) Condiciones KKT (Análiticas a Mano)**
Debemos construir el Lagrangiano del problema para las derivadas:
$$ L(x, \mu) = f(x) + \sum_{i=1}^{6} \mu_i g_i(x) $$

**Ecuaciones de Estacionariedad:**
$$ \frac{\partial L}{\partial x_1} = 4x_1 - 80 + \mu_1 + \mu_2 + 2\mu_3 + \mu_4 - \mu_5 = 0 $$
$$ \frac{\partial L}{\partial x_2} = 2x_2 - 100 + \mu_1 + 3\mu_2 + \mu_3 - \mu_6 = 0 $$

Estas se unen a:
- Factibilidades triviales: Todas las $g_i(x) \le 0$.
- Holguras Complementarias: Multiplicar las restricciones por su lagrangiano ($\mu_i g_i(x) = 0$).

*(Demostración en pizarra del óptimo a mano)*
Para evaluar analíticamente el hipotético óptimo $(x_1=15, x_2=25)$:
Calculamos cuáles $g(x)$ se vuelven cero y notamos que se cumplen exactamente con igualdad (activas) $g_1$ y $g_2$ ($15+25=40$ y $15+75=90$). 
Como $g_3, g_4, g_5, g_6$ son estrictamente negativas (no activas), su multiplicador es $0$ vía Holgura: $\mu_3 = \mu_4 = \mu_5 = \mu_6 = 0$.

Reemplazando en el sistema de dos derivadas te quedan 2 variables $(\mu_1, \mu_2)$ :
*   $4(15) - 80 + \mu_1 + \mu_2 = 0 \Rightarrow \mu_1 + \mu_2 = 20$
*   $2(25) - 100 + \mu_1 + 3\mu_2 = 0 \Rightarrow \mu_1 + 3\mu_2 = 50$
Esto arroja $\mu_1 = 5$ y $\mu_2 = 15$. ¡Ambos son positivos $\ge 0$! Se verifican todas las condiciones analíticamente para determinar que sí es un mínimo estándar (que es el máximo original de Utilidad) por tratarse de un problema convexo de la forma estándar.

#### **c) Resolución Computacional**

In [ ]:
import numpy as np
from scipy.optimize import minimize

# 1. FUNCIÓN OBJETIVO
# Pasamos la función en la forma estándar (minimizar f(x)) 
fo = lambda x: 2*(x[0]**2) - 80*x[0] + (x[1]**2) - 100*x[1]

# 2. RESTRICCIONES
# Cuidado!: Al utilizar el engine 'SLSQP' de scipy.optimize.minimize, 
# si le declaras tipo 'ineq', lo interpreta como una restricción mayor o igual ( g(x) >= 0 ). 
# Por eso los lambdas los debemos multiplicar por -1 con respecto a nuestra forma analítica g(x) <= 0.
r1 = lambda x: -(x[0] + x[1] - 40)
r2 = lambda x: -(x[0] + 3*x[1] - 90)
r3 = lambda x: -(2*x[0] + x[1] - 65)
r4 = lambda x: -(x[0] - 30)

restricciones = [
    {'type': 'ineq', 'fun': r1},
    {'type': 'ineq', 'fun': r2},
    {'type': 'ineq', 'fun': r3},
    {'type': 'ineq', 'fun': r4}
]

# 3. NO NEGATIVIDAD
limites = [(0, None), (0, None)]

# 4. RESOLUCIÓN
x0 = np.array([0, 0])

# Método: Sequential Least SQuares Programming (SLSQP), ideal para problemas acotados o desiguales
resultado = minimize(fo, x0, method='SLSQP', bounds=limites, constraints=restricciones)

print("----- RESULTADOS MÁXIMA UTILIDAD -----")
if resultado.success:
    print(f"Estado de optimización: {resultado.message}")
    print(f"Servidores Media (x1):   {resultado.x[0]:.2f} unidades")
    print(f"Servidores Altas (x2):   {resultado.x[1]:.2f} unidades")
    # Para imprimir la Utilidad MÁXIMA (original), usamos el negativo o multiplicamos por -1 
    print(f"Utilidad Máxima Proyectada: {-resultado.fun:.2f} (miles de USD)")
else:
    print(f"Falló la convergencia: {resultado.message}")

# Podemos evaluar las funciones de restricción (en formato g(x) <= 0) para ver cuáles hacen cuello de botella
print("\n-- INFORME DE HOLGURAS (slacks / restricciones activas) --")
print(f"Holgura g1: {-r1(resultado.x):.4f} (<= 0) -->", "ACTIVA" if np.abs(r1(resultado.x)) < 1e-4 else "Inactiva")
print(f"Holgura g2: {-r2(resultado.x):.4f} (<= 0) -->", "ACTIVA" if np.abs(r2(resultado.x)) < 1e-4 else "Inactiva")
print(f"Holgura g3: {-r3(resultado.x):.4f} (<= 0) -->", "ACTIVA" if np.abs(r3(resultado.x)) < 1e-4 else "Inactiva")
print(f"Holgura g4: {-r4(resultado.x):.4f} (<= 0) -->", "ACTIVA" if np.abs(r4(resultado.x)) < 1e-4 else "Inactiva")

----- RESULTADOS MÁXIMA UTILIDAD -----
Estado de optimización: Optimization terminated successfully
Servidores Media (x1):   15.00 unidades
Servidores Altas (x2):   25.00 unidades
Utilidad Máxima Proyectada: 2625.00 (miles de USD)

-- INFORME DE HOLGURAS (slacks / restricciones activas) --
Holgura g1: 0.0000 (<= 0) --> ACTIVA
Holgura g2: 0.0000 (<= 0) --> ACTIVA
Holgura g3: -10.0000 (<= 0) --> Inactiva
Holgura g4: -15.0000 (<= 0) --> Inactiva
